# 🛡️ PEDAS 2026 — Feature Engineering & Modeling Pipeline (v3)
**Platform Evaluasi Data Sains (PEDAS) 2026**  
*Pipeline: Data Ingestion ➔ Typo Normalization ➔ Multimodal Mislabel Audit ➔ Zero-Leakage Confidence Recomputation ➔ Rare Class Handling ➔ Infrastructure & Temporal Feature Extraction ➔ Quality Assurance & Stratified Baseline Modeling*

---

### 📋 Ringkasan Alur Pipeline v3:
1. **Setup & Ingestion**: Inisialisasi modul audit mislabeling dan dataset mentah (`training.csv` & `predict.csv`).
2. **Label Standardization**: Normalisasi variasi penulisan target kategori (typo & casing).
3. **Mislabel Audit**: Audit anomali label train berbasis bukti gabungan (URL keyword, IP exact, subnet `/24`, & domain).
4. **Confidence Level Recomputation**: Rekalkulasi kepercayaan berbasis statistik referensi train (*zero data leakage*).
5. **Relabeling Execution**: Penerapan relabel otomatis untuk sampel dengan `suspect_score >= 5` dan pencatatan diff.
6. **Rare Class Consolidation**: Penggabungan kategori sangat langka (< 5 baris) ke kategori `other`.
7. **Feature Engineering**:
   - Skalasi `confidence_norm` (0–1)
   - Fitur temporal: `discovered_hour`, `discovered_dayofweek`
   - Umur domain & anomali tanggal negatif: `domain_age_days`, `domain_age_is_negative`
   - Fitur jaringan: `ip_frequency`, `ip_subnet_frequency`, `ip_is_missing`
   - Fitur URL: `url_length`, `url_has_base64`
   - Fitur brand: `brand_is_missing`
   - Frekuensi domain & One-Hot Encoding `sld` (diselaraskan train & test)
8. **Sanity Check & Export**: Validasi missing values dan penyimpanan dataset bersih ke `data/result/v3/`.
9. **Baseline Evaluation**: Validasi model awal menggunakan Stratified Random Forest Classifier.


## 1. Setup Lingkungan & Import Library
Memuat library manipulasi data tabular serta modul `audit_mislabel.py` dari parent folder (`../`).


In [1]:
# ============================================================
# CELL 1: Setup & Import
# ============================================================
import sys
import os
import re
import pandas as pd

# Pastikan modul parent directory (audit_mislabel.py) terdeteksi
sys.path.insert(0, '../')

from audit_mislabel import (
    audit_mislabel, apply_relabel, summarize_audit, summarize_diff,
    recompute_confidence, score_confidence_evidence, get_reference_stats
)

pd.set_option('display.max_columns', None)


## 2. Pemuatan Dataset (Train & Test)
Memuat dataset mentah dari direktori `../../data/raw/`:
- `training.csv`: Data latih lengkap dengan label `category`.
- `predict.csv`: Data uji evaluasi tanpa label target.


In [2]:
# ============================================================
# CELL 2: Ambil data
# ============================================================
train = pd.read_csv('../../data/raw/training.csv')
test  = pd.read_csv('../../data/raw/predict.csv')

print(f"Dimensi Training : {train.shape}")
print(f"Dimensi Predict  : {test.shape}")
train.head(3)


Dimensi Training : (8400, 10)
Dimensi Predict  : (1500, 10)


,url,brand,discovered,confidence_level,ip,domain,sld,category,registrar,registration_date
0,https://lucah3.***********.my.id/,Telegram,5/1/2024 21:52,100,NaN,***********.my.id,my.id,phishingg,PT Web Media Technology Indonesia,5/14/2026
1,https://dpmptsp.*********.go.id/petapotensi/da...,judi online,8/5/2024 20:41,100,103.162.68.84,*********.go.id,go.id,online gambling,Kementerian Komunikasi dan Informatika,5/11/2009
2,http://klikpad.bkpd.**************.go.id/klikp...,-,6/14/2024 7:05,100,103.18.117.8,**************.go.id,go.id,online gambling,Kementerian Komunikasi dan Informatika,3/13/2008


## 3. Pembersihan & Standarisasi Penulisan Label Kategori
Melakukan normalisasi penulisan teks kategori target (`category`) pada data latih untuk mengatasi variasi *typo* dan perbedaan kapitalisasi.

> ⚠️ **Catatan**: Pembersihan target hanya dilakukan pada **TRAIN** karena dataset **TEST** (`predict.csv`) tidak memiliki kolom target.


In [3]:
# ============================================================
# CELL 3: Bersihkan penulisan category (TRAIN saja - test gak punya category)
# ============================================================
cat_map = {
    'online gamblingg': 'online gambling', 'Online Gambling': 'online gambling',
    'phishingg': 'phishing', 'otherr': 'other', 'Other': 'other',
    'malwaree': 'malware', 'spamm': 'spam', 'Brand': 'brand',
    'FakeShop': 'fakeshop', 'PIIExposure': 'pii_exposure',
}
train['category'] = train['category'].str.strip().replace(cat_map).str.lower().str.strip()

print("=== Distribusi Kategori Setelah Pembersihan ===")
print(train['category'].value_counts())


=== Distribusi Kategori Setelah Pembersihan ===
category
online gambling    5447
phishing           2253
other               284
spam                185
malware             179
brand                45
fakeshop              5
violence              1
pii_exposure          1
Name: count, dtype: int64


## 4. Audit Mislabeling pada Data Latih
Mengidentifikasi potensi kesalahan pelabelan (*mislabeling*) pada training set dengan memeriksa bukti pendukung multivariat:
- **Keyword URL**: Kemunculan kata kunci khas kategori tertentu pada URL.
- **IP Exact & Subnet /24 Dominance**: Kemurnian (*purity*) dan frekuensi kemunculan IP / subnet tertentu pada kategori tertentu.
- **Domain Dominance**: Dominasi kategori per nama domain.


In [4]:
# ============================================================
# CELL 4: Audit & perbaiki category (TRAIN saja, pakai bukti url + ip)
# ============================================================
train_audited = audit_mislabel(train)
summarize_audit(train_audited)


=== Distribusi skor kecurigaan ===
suspect_score
11       1
8       24
7        1
6        1
5       34
4       10
3      129
2      134
1      132
0     7934
Name: count, dtype: int64

Skor >=5 (relabel otomatis)      : 61 baris
Skor 3-4 (verifikasi manual)     : 139 baris
Skor 1-2 (biarkan, terlalu lemah): 266 baris


## 5. Rekalkulasi & Perbaikan Anomali `confidence_level`
Memperbaiki nilai `confidence_level` yang inkonsisten dengan bukti aktual:
- **Train**: Menggunakan metadata bukti yang telah dihitung dari proses `audit_mislabel()`.
- **Test**: Menghitung skor bukti murni menggunakan statistik referensi dari data latih (`ref_ip_stats`, `ref_subnet_stats`) untuk menjamin **Zero Data Leakage**.


In [5]:
# ============================================================
# CELL 5: Perbaiki confidence_level anomali - TRAIN & TEST
# - Train: pakai bukti dari audit_mislabel (sudah punya url, ip dominant dsb)
# - Test: gak punya category, tapi BISA pakai referensi statistik dari TRAIN
# (misal IP test sering muncul di train dan 100% phishing -> test juga kuat)
# ============================================================

# Hitung referensi (purity dsb) HANYA dari train, sekali saja
ref_ip_stats, ref_subnet_stats = get_reference_stats(train_audited)

# Train: sudah punya semua kolom bukti dari audit_mislabel() -> langsung recompute
train_audited = recompute_confidence(train_audited)

# Test: gak punya 'category', pakai referensi dari train
test_scored = score_confidence_evidence(test, ref_ip_stats, ref_subnet_stats)
test = recompute_confidence(test_scored)

print("Distribusi confidence_level train setelah diperbaiki:")
print(train_audited['confidence_level'].value_counts().sort_index())
print("\nDistribusi confidence_level test setelah diperbaiki:")
print(test['confidence_level'].value_counts().sort_index())


Distribusi confidence_level train setelah diperbaiki:
confidence_level
0        16
40       18
50       23
60        3
90      209
100    8131
Name: count, dtype: int64

Distribusi confidence_level test setelah diperbaiki:
confidence_level
50        4
90       43
100    1453
Name: count, dtype: int64


## 6. Penerapan Relabeling Otomatis & Pencatatan Log
Menerapkan penggantian label kategori untuk baris-baris pada training set yang memiliki skor kecurigaan tinggi (`suspect_score >= 5`).
Perubahan dicatat dan diekspor ke `relabel_diff.csv`.


In [6]:
# ============================================================
# CELL 6: Terapkan relabel category (TRAIN saja, skor >= 5)
# ============================================================
train, diff_table = apply_relabel(train_audited, score_threshold=5)
summarize_diff(diff_table)

os.makedirs('../../data/result/v3', exist_ok=True)
diff_table.to_csv('../../data/result/v3/relabel_diff.csv', index=False)
print("Log relabeling disimpan di data/result/v3/relabel_diff.csv")


=== Total baris yang benar-benar berubah label: 61 ===

=== Perubahan per pasangan (label lama -> label baru) ===
category_before  category_after 
phishing         online gambling    20
malware          online gambling    14
spam             online gambling    12
other            online gambling     8
online gambling  phishing            6
spam             phishing            1
dtype: int64
Log relabeling disimpan di data/result/v3/relabel_diff.csv


## 7. Penanganan Kategori Sangat Langka (*Rare Classes*)
Menggabungkan kategori yang memiliki frekuensi kurang dari 5 baris ke kategori `other` untuk menghindari masalah saat pembagian lipatan validasi silang (cross-validation).


In [7]:
# ============================================================
# CELL 7: Gabung kategori super langka (<5 baris) ke 'other'
# ============================================================
category_counts = train['category'].value_counts()
rare_categories = category_counts[category_counts < 5].index.tolist()
print("Kategori digabung ke 'other':", rare_categories)

train['category'] = train['category'].replace({cat: 'other' for cat in rare_categories})
print("\nDistribusi category final:")
print(train['category'].value_counts())


Kategori digabung ke 'other': ['violence', 'pii_exposure']

Distribusi category final:
category
online gambling    5495
phishing           2240
other               278
spam                172
malware             165
brand                45
fakeshop              5
Name: count, dtype: int64


## 8. Normalisasi Skala `confidence_level`
Menskalakan nilai `confidence_level` dari rentang persentase $[0, 100]$ menjadi rentang $[0.0, 1.0]$ (`confidence_norm`) pada dataset train dan test.


In [8]:
# ============================================================
# CELL 8: Normalisasi confidence_level ke 0-1 - TRAIN & TEST
# ============================================================
train['confidence_norm'] = train['confidence_level'] / 100
test['confidence_norm'] = test['confidence_level'] / 100


## 9. Rekayasa Fitur Waktu (Temporal / Datetime)
Mengonversi kolom string tanggal ke objek `datetime` dan mengekstrak komponen waktu siklikal:
- `discovered_hour`: Jam saat URL pertama kali ditemukan (0–23).
- `discovered_dayofweek`: Hari dalam sepekan (0 = Senin, 6 = Minggu).


In [9]:
# ============================================================
# CELL 9: Ubah format tanggal - TRAIN & TEST
# ============================================================
train['discovered'] = pd.to_datetime(train['discovered'], format='%m/%d/%Y %H:%M')
train['registration_date'] = pd.to_datetime(train['registration_date'], format='%m/%d/%Y')
test['discovered'] = pd.to_datetime(test['discovered'], format='%m/%d/%Y %H:%M')
test['registration_date'] = pd.to_datetime(test['registration_date'], format='%m/%d/%Y')

train['discovered_hour'] = train['discovered'].dt.hour
train['discovered_dayofweek'] = train['discovered'].dt.dayofweek
test['discovered_hour'] = test['discovered'].dt.hour
test['discovered_dayofweek'] = test['discovered'].dt.dayofweek


## 10. Rekayasa Fitur Usia Domain & Anomali Negatif
Menghitung usia domain dalam hari (`domain_age_days = discovered - registration_date`).
- Nilai negatif dipertahankan sebagai sinyal kuat indikator fraud/phishing melalui fitur biner `domain_age_is_negative`.
- Nilai `NaN` pada usia domain diimputasi menggunakan nilai median dari masing-masing dataset.


In [10]:
# ============================================================
# CELL 10: Selisih discovered vs registration_date - TRAIN & TEST
# ============================================================
train['domain_age_days'] = (train['discovered'] - train['registration_date']).dt.days
test['domain_age_days'] = (test['discovered'] - test['registration_date']).dt.days

train['domain_age_is_negative'] = (train['domain_age_days'] < 0).astype(int)
test['domain_age_is_negative'] = (test['domain_age_days'] < 0).astype(int)

train['domain_age_days'] = train['domain_age_days'].fillna(train['domain_age_days'].median())
test['domain_age_days'] = test['domain_age_days'].fillna(train['domain_age_days'].median())


## 11. Rekayasa Fitur Infrastruktur Jaringan (IP & Subnet)
Mengekstrak frekuensi kemunculan IP dan subnet `/24`.
> 🛡️ **Aturan Anti-Leakage**: Kamus frekuensi (`ip_freq_map` dan `subnet_freq_map`) **wajib** dihitung hanya dari **TRAIN**, kemudian dipetakan ke train dan test.


In [11]:
# ============================================================
# CELL 11: Uraikan IP - TRAIN & TEST (hitung referensi HANYA dari train)
# ============================================================
ip_freq_map = train['ip'].value_counts().to_dict()

train['ip_prefix24'] = train['ip'].str.rsplit('.', n=1).str[0]
test['ip_prefix24'] = test['ip'].str.rsplit('.', n=1).str[0]
subnet_freq_map = train['ip_prefix24'].value_counts().to_dict()

train['ip_frequency'] = train['ip'].map(ip_freq_map).fillna(0)
train['ip_subnet_frequency'] = train['ip_prefix24'].map(subnet_freq_map).fillna(0)
train['ip_is_missing'] = train['ip'].isna().astype(int)

test['ip_frequency'] = test['ip'].map(ip_freq_map).fillna(0)
test['ip_subnet_frequency'] = test['ip_prefix24'].map(subnet_freq_map).fillna(0)
test['ip_is_missing'] = test['ip'].isna().astype(int)

train = train.drop(columns=['ip_prefix24'])
test = test.drop(columns=['ip_prefix24'])


## 12. Rekayasa Fitur Leksikal URL
Mengekstrak panjang URL dan mendeteksi string terenkode Base64 (panjang $\ge 40$ karakter) yang sering diasosiasikan dengan token serangan atau malware payload.


In [12]:
# ============================================================
# CELL 12: Fitur dari URL - TRAIN & TEST
# ============================================================
train['url_length'] = train['url'].str.len()
test['url_length'] = test['url'].str.len()

train['url_has_base64'] = train['url'].str.contains(r'[A-Za-z0-9+/_-]{40,}={0,2}', regex=True).astype(int)
test['url_has_base64'] = test['url'].str.contains(r'[A-Za-z0-9+/_-]{40,}={0,2}', regex=True).astype(int)


## 13. Rekayasa Fitur Entitas Merek (*Brand Missingness*)
Menandai sampel yang tidak memiliki asosiasi brand resmi (`brand_is_missing`), termasuk penanda karakter placeholder seperti `-`.


In [13]:
# ============================================================
# CELL 13: Brand -- flag missing (termasuk '-') - TRAIN & TEST
# ============================================================
for df in [train, test]:
    df['brand_is_missing'] = (df['brand'].isna() | (df['brand'].str.strip() == '-')).astype(int)


## 14. Rekayasa Fitur Domain & One-Hot Encoding SLD
1. **Domain Frequency**: Menghitung seberapa sering nama domain muncul di dataset latih.
2. **One-Hot Encoding SLD**: Melakukan enkoding kategori SLD (*second-level domain*) dan menyelaraskan kolom biner antara data train dan test agar memiliki dimensi matriks yang identik.


In [14]:
# ============================================================
# CELL 14a: Domain frequency - TRAIN & TEST
# ============================================================
domain_freq_map = train['domain'].value_counts().to_dict()
train['domain_frequency'] = train['domain'].map(domain_freq_map).fillna(0)
test['domain_frequency'] = test['domain'].map(domain_freq_map).fillna(0)

# ============================================================
# CELL 14b: One-hot encoding sld - TRAIN & TEST
# ============================================================
train = pd.get_dummies(train, columns=['sld'], prefix='sld')
test = pd.get_dummies(test, columns=['sld'], prefix='sld')

train_sld_cols = set(c for c in train.columns if c.startswith('sld_'))
test_sld_cols = set(c for c in test.columns if c.startswith('sld_'))

for col in train_sld_cols - test_sld_cols:
    test[col] = 0
for col in test_sld_cols - train_sld_cols:
    train[col] = 0


## 15. Pemeriksaan Kualitas Data (Sanity Check) & Ekspor Dataset Bersih
Memeriksa seluruh fitur input model untuk memastikan tidak ada nilai `NaN` yang tersisa, kemudian menyimpan dataset final ke folder `../../data/result/v3/clean/` dan `../../data/result/v3/`.


In [ ]:
# ============================================================
# CELL 15: Cek sanity & simpan hasil cleaning
# ============================================================
sld_cols = [c for c in train.columns if c.startswith('sld_')]
feature_cols = [
    'confidence_norm', 'discovered_hour', 'discovered_dayofweek',   
    'domain_age_days', 'domain_age_is_negative',
    'ip_frequency', 'ip_subnet_frequency', 'ip_is_missing',
    'url_length', 'url_has_base64', 'brand_is_missing', 'domain_frequency',
] + sld_cols

print("Missing value tersisa (train):")
print(train[feature_cols].isna().sum()[train[feature_cols].isna().sum() > 0])
print("\nMissing value tersisa (test):")
print(test[feature_cols].isna().sum()[test[feature_cols].isna().sum() > 0])

# Buat folder penyimpanan
os.makedirs('../../data/result/v3/clean', exist_ok=True)

# Simpan ke v3/clean dan v3 root
train.to_csv('../../data/result/v3/clean/train_final.csv', index=False)
test.to_csv('../../data/result/v3/clean/test_final.csv', index=False)
train.to_csv('../../data/result/v3/train_final.csv', index=False)
test.to_csv('../../data/result/v3/test_final.csv', index=False)

print("\nDataset berhasil disimpan ke:")
print("  - ../../data/result/v3/clean/train_final.csv")
print("  - ../../data/result/v3/clean/test_final.csv")


Missing value tersisa (train):
Series([], dtype: int64)

Missing value tersisa (test):
Series([], dtype: int64)

Dataset berhasil disimpan ke:
  - ../../data/result/v3/clean/train_final.csv
  - ../../data/result/v3/clean/test_final.csv


## 16. Eksperimen Validasi Model Baseline (Internal Split)
Menguji performa fitur rekayasa menggunakan model *Random Forest Classifier* dengan pembagian data 80% train dan 20% validation secara stratified.

> 🔒 **Penting**: Eksperimen ini murni evaluasi internal pada data train dan tidak menyentuh dataset `predict.csv`.


In [16]:
# ============================================================
# CELL 16: Split train/validation & training baseline model
# (CUMA EKSPERIMEN INTERNAL, TIDAK menyentuh test.csv/predict.csv)
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X = train[feature_cols]
y = train['category']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

print(classification_report(y_val, model.predict(X_val), zero_division=0))


                 precision    recall  f1-score   support

          brand       1.00      0.56      0.71         9
       fakeshop       0.00      0.00      0.00         1
        malware       0.70      0.42      0.53        33
online gambling       0.97      0.99      0.98      1099
          other       0.98      0.96      0.97        56
       phishing       0.98      0.97      0.97       448
           spam       0.79      0.88      0.83        34

       accuracy                           0.97      1680
      macro avg       0.77      0.68      0.71      1680
   weighted avg       0.96      0.97      0.96      1680

